# SUB006 -- Fragment Assembly + Per-Chain Template Prediction

**Competition**: Stanford RNA 3D Folding Part 2  
**Metric**: TM-score (higher is better, best-of-5)  
**Lineage**: SUB004 (0.211) -> SUB005 (pending) -> SUB006

Key improvements over SUB004/SUB005:
- Per-chain independent template matching (not full-complex matching)
- Fragment-based assembly for long chains without good full-length templates
- Coverage-weighted template scoring
- Multi-strategy diversity for best-of-5 predictions
- RNA-specific alignment scoring (transition/transversion)
- Improved gap interpolation with helical geometry
- Kabsch-based template blending and fragment stitching

In [ ]:
import os
import sys
import time
import zlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 60)
print("SUB006: Fragment Assembly + Per-Chain Template Prediction")
print("=" * 60)

INPUT_BASE = "/kaggle/input"
if os.path.isdir(INPUT_BASE):
    print(f"\nContents of {INPUT_BASE}:")
    for item in sorted(os.listdir(INPUT_BASE)):
        full = os.path.join(INPUT_BASE, item)
        if os.path.isdir(full):
            sub_items = os.listdir(full)
            print(f"  {item}/ ({len(sub_items)} items)")
            for si in sorted(sub_items)[:10]:
                print(f"    {si}")
        else:
            print(f"  {item} ({os.path.getsize(full)} bytes)")

In [ ]:
# ============================================================
# Data Loading
# ============================================================
CANDIDATE_BASES = [
    "/kaggle/input/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-part-2",
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
]

DATA_PATH = None
for p in CANDIDATE_BASES:
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, "test_sequences.csv")):
        DATA_PATH = p
        break

if DATA_PATH is None and os.path.isdir(INPUT_BASE):
    for root, dirs, files in os.walk(INPUT_BASE):
        if "test_sequences.csv" in files:
            DATA_PATH = root
            break

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find competition data. "
        f"Searched: {CANDIDATE_BASES}. "
        f"Available: {os.listdir(INPUT_BASE) if os.path.isdir(INPUT_BASE) else 'N/A'}"
    )

print(f"\nUsing data path: {DATA_PATH}")
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

train_seqs = pd.read_csv(os.path.join(DATA_PATH, "train_sequences.csv"))
test_seqs = pd.read_csv(os.path.join(DATA_PATH, "test_sequences.csv"))
train_labels = pd.read_csv(os.path.join(DATA_PATH, "train_labels.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_PATH, "sample_submission.csv"))

val_seq_path = os.path.join(DATA_PATH, "validation_sequences.csv")
val_lab_path = os.path.join(DATA_PATH, "validation_labels.csv")
HAS_VAL = os.path.isfile(val_seq_path) and os.path.isfile(val_lab_path)
if HAS_VAL:
    val_seqs = pd.read_csv(val_seq_path)
    val_labels = pd.read_csv(val_lab_path)
    print(f"Validation set loaded: {len(val_seqs)} targets")
else:
    print("No validation set found")

print(f"Train sequences: {len(train_seqs)}")
print(f"Test sequences:  {len(test_seqs)}")
print(f"Train labels:    {len(train_labels)}")
print(f"Sample sub:      {len(sample_sub)}")

In [ ]:
# ============================================================
# Build Template Library from Training Data
# ============================================================
SENTINEL = -1e18

def process_labels(labels_df):
    coords_dict = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for id_prefix, group in labels_df.groupby(prefixes, sort=False):
        sorted_group = group.sort_values("resid")
        c = sorted_group[["x_1", "y_1", "z_1"]].values.astype(np.float64)
        mask = np.all(np.abs(c - SENTINEL) < 1e10, axis=1)
        if mask.any():
            c[mask] = np.nan
        coords_dict[id_prefix] = c
    return coords_dict

print("Processing training labels...")
t0 = time.time()
train_coords_dict = process_labels(train_labels)
print(f"Processed {len(train_coords_dict)} targets in {time.time()-t0:.1f}s")

TRAIN_IDS, TRAIN_SEQS, TRAIN_COORDS, TRAIN_LENS = [], [], [], []
TRAIN_VALID_FRAC = []  # fraction of residues with valid coords

for r in train_seqs.itertuples(index=False):
    tid = r.target_id
    if tid not in train_coords_dict:
        continue
    seq = r.sequence
    coords = train_coords_dict[tid]
    if len(coords) != len(seq):
        continue
    valid_mask = ~np.isnan(coords[:, 0])
    n_valid = valid_mask.sum()
    if n_valid < 5:
        continue
    if np.all(coords[valid_mask] == 0):
        continue
    TRAIN_IDS.append(tid)
    TRAIN_SEQS.append(seq)
    TRAIN_COORDS.append(coords)
    TRAIN_LENS.append(len(seq))
    TRAIN_VALID_FRAC.append(float(n_valid) / len(seq))

TRAIN_LENS = np.array(TRAIN_LENS, dtype=np.int32)
TRAIN_VALID_FRAC = np.array(TRAIN_VALID_FRAC, dtype=np.float32)
print(f"Template library: {len(TRAIN_IDS)} templates with valid coordinates")
print(f"  Length range: {TRAIN_LENS.min()}-{TRAIN_LENS.max()}, mean={TRAIN_LENS.mean():.0f}")
print(f"  Valid fraction: mean={TRAIN_VALID_FRAC.mean():.2f}, median={np.median(TRAIN_VALID_FRAC):.2f}")

In [ ]:
# ============================================================
# K-mer Indexing for Fast Template Retrieval
# ============================================================
KMER_K = 5
PREFILTER_TOP = 300
ALIGN_TOP = 30

_BASE2 = {"A": 0, "C": 1, "G": 2, "U": 3}

def kmer_set_2bit(seq, k=KMER_K):
    if len(seq) < k:
        return frozenset()
    mask = (1 << (2 * k)) - 1
    code = 0
    out = set()
    valid = True
    for i in range(k):
        b = _BASE2.get(seq[i])
        if b is None:
            valid = False
            break
        code = ((code << 2) | b) & mask
    if valid:
        out.add(code)
        for i in range(k, len(seq)):
            b = _BASE2.get(seq[i])
            if b is None:
                continue
            code = ((code << 2) | b) & mask
            out.add(code)
    return frozenset(out)

print("Building k-mer index...")
t0 = time.time()
TRAIN_KMERS = [kmer_set_2bit(s, KMER_K) for s in TRAIN_SEQS]
print(f"K-mer index built in {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# Alignment Functions with RNA-specific scoring
# ============================================================
_MATCH = 2
_MISMATCH_TRANSITION = -0.5  # A<->G, C<->U (purines/pyrimidines)
_MISMATCH_TRANSVERSION = -1.5
_GAP_OPEN = -4
_GAP_EXTEND = -1

_PURINES = {'A', 'G'}
_PYRIMIDINES = {'C', 'U'}

def _mismatch_score(a, b):
    if a == b:
        return _MATCH
    if (a in _PURINES and b in _PURINES) or (a in _PYRIMIDINES and b in _PYRIMIDINES):
        return _MISMATCH_TRANSITION
    return _MISMATCH_TRANSVERSION


def needleman_wunsch(seq_a, seq_b):
    """Affine gap NW alignment with RNA-specific scoring."""
    n, m = len(seq_a), len(seq_b)
    if n * m > 25_000_000:
        return _banded_nw(seq_a, seq_b, band_width=150)
    
    dp = np.full((n + 1, m + 1), -999999.0, dtype=np.float32)
    dp[0, 0] = 0.0
    for j in range(1, m + 1):
        dp[0, j] = _GAP_OPEN + (j - 1) * _GAP_EXTEND
    for i in range(1, n + 1):
        dp[i, 0] = _GAP_OPEN + (i - 1) * _GAP_EXTEND
    
    b_arr = np.array(list(seq_b))
    
    for i in range(1, n + 1):
        ch_a = seq_a[i - 1]
        scores = np.array([_mismatch_score(ch_a, ch_b) for ch_b in b_arr], dtype=np.float32)
        diag = dp[i - 1, :-1] + scores
        up = dp[i - 1, 1:] + _GAP_EXTEND
        dp[i, 1:] = np.maximum(diag, up)
        for j in range(1, m + 1):
            left = dp[i, j - 1] + _GAP_EXTEND
            dp[i, j] = max(dp[i, j], left)
    
    map_a_to_b = {}
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = _mismatch_score(seq_a[i-1], seq_b[j-1])
            if abs(dp[i, j] - (dp[i-1, j-1] + s)) < 1e-3:
                map_a_to_b[i - 1] = j - 1
                i -= 1
                j -= 1
                continue
        if i > 0 and abs(dp[i, j] - (dp[i-1, j] + _GAP_EXTEND)) < 1e-3:
            i -= 1
        elif j > 0:
            j -= 1
        else:
            break
    
    return map_a_to_b, float(dp[n, m])


def _banded_nw(seq_a, seq_b, band_width=150):
    n, m = len(seq_a), len(seq_b)
    NEGINF = -999999.0
    dp = {}
    dp[(0, 0)] = 0.0
    for j in range(1, min(band_width + 1, m + 1)):
        dp[(0, j)] = _GAP_OPEN + (j - 1) * _GAP_EXTEND
    for i in range(1, min(band_width + 1, n + 1)):
        dp[(i, 0)] = _GAP_OPEN + (i - 1) * _GAP_EXTEND
    
    ratio = m / n if n > 0 else 1.0
    
    for i in range(1, n + 1):
        center_j = int(i * ratio)
        j_lo = max(1, center_j - band_width)
        j_hi = min(m, center_j + band_width)
        for j in range(j_lo, j_hi + 1):
            s = _mismatch_score(seq_a[i-1], seq_b[j-1])
            d = dp.get((i-1, j-1), NEGINF) + s
            u = dp.get((i-1, j), NEGINF) + _GAP_EXTEND
            l = dp.get((i, j-1), NEGINF) + _GAP_EXTEND
            dp[(i, j)] = max(d, u, l)
    
    score = dp.get((n, m), NEGINF)
    
    map_a_to_b = {}
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = _mismatch_score(seq_a[i-1], seq_b[j-1])
            if abs(dp.get((i, j), NEGINF) - (dp.get((i-1, j-1), NEGINF) + s)) < 1e-3:
                map_a_to_b[i - 1] = j - 1
                i -= 1
                j -= 1
                continue
        if i > 0 and abs(dp.get((i, j), NEGINF) - (dp.get((i-1, j), NEGINF) + _GAP_EXTEND)) < 1e-3:
            i -= 1
        elif j > 0:
            j -= 1
        else:
            break
    
    return map_a_to_b, float(score)


def nw_score_only(seq_a, seq_b):
    n, m = len(seq_a), len(seq_b)
    if n * m > 50_000_000:
        return _kmer_score_proxy(seq_a, seq_b)
    
    prev = np.full(m + 1, -999999.0, dtype=np.float32)
    prev[0] = 0.0
    for j in range(1, m + 1):
        prev[j] = _GAP_OPEN + (j - 1) * _GAP_EXTEND
    b_arr = list(seq_b)
    
    for i in range(1, n + 1):
        curr = np.empty(m + 1, dtype=np.float32)
        curr[0] = _GAP_OPEN + (i - 1) * _GAP_EXTEND
        ch_a = seq_a[i - 1]
        scores = np.array([_mismatch_score(ch_a, ch_b) for ch_b in b_arr], dtype=np.float32)
        diag = prev[:-1] + scores
        up = prev[1:] + _GAP_EXTEND
        curr[1:] = np.maximum(diag, up)
        for j in range(1, m + 1):
            curr[j] = max(curr[j], curr[j - 1] + _GAP_EXTEND)
        prev = curr
    
    return float(prev[m])


def _kmer_score_proxy(seq_a, seq_b, k=5):
    ka = kmer_set_2bit(seq_a, k)
    kb = kmer_set_2bit(seq_b, k)
    if not ka or not kb:
        return 0.0
    inter = len(ka & kb)
    union = len(ka | kb)
    jaccard = inter / union if union > 0 else 0.0
    return jaccard * 2.0 * min(len(seq_a), len(seq_b))

print("Alignment functions loaded.")

In [ ]:
# ============================================================
# Kabsch Superposition and Coordinate Transfer
# ============================================================
def kabsch_superpose(P, Q):
    """Kabsch algorithm: find optimal rotation+translation to align P onto Q."""
    assert P.shape == Q.shape and P.shape[1] == 3
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    P_c = P - centroid_P
    Q_c = Q - centroid_Q
    H = P_c.T @ Q_c
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    sign_matrix = np.diag([1.0, 1.0, np.sign(d)])
    R = Vt.T @ sign_matrix @ U.T
    t = centroid_Q - R @ centroid_P
    return R, t


def transfer_coordinates(query_seq, template_seq, template_coords, alignment_map):
    qlen = len(query_seq)
    result = np.full((qlen, 3), np.nan, dtype=np.float64)
    
    n_transferred = 0
    for q_pos, t_pos in alignment_map.items():
        if q_pos < qlen and t_pos < len(template_coords):
            c = template_coords[t_pos]
            if not np.any(np.isnan(c)):
                result[q_pos] = c
                n_transferred += 1
    
    coverage = n_transferred / qlen if qlen > 0 else 0
    
    mapped = sorted([k for k in range(qlen) if not np.isnan(result[k, 0])])
    if not mapped:
        return _helical_chain(qlen), 0.0
    
    for i in range(qlen):
        if not np.isnan(result[i, 0]):
            continue
        prev_v = next((j for j in range(i - 1, -1, -1) if not np.isnan(result[j, 0])), -1)
        next_v = next((j for j in range(i + 1, qlen) if not np.isnan(result[j, 0])), -1)
        if prev_v >= 0 and next_v >= 0:
            gap_len = next_v - prev_v
            pos_in_gap = i - prev_v
            w = pos_in_gap / gap_len
            base = (1 - w) * result[prev_v] + w * result[next_v]
            if gap_len > 2:
                direction = result[next_v] - result[prev_v]
                norm_d = np.linalg.norm(direction)
                if norm_d > 1e-6:
                    perp = np.cross(direction, [0, 0, 1])
                    perp_norm = np.linalg.norm(perp)
                    if perp_norm < 1e-6:
                        perp = np.cross(direction, [0, 1, 0])
                        perp_norm = np.linalg.norm(perp)
                    perp = perp / (perp_norm + 1e-12)
                    perp2 = np.cross(direction / norm_d, perp)
                    angle = 2 * np.pi * pos_in_gap / max(gap_len, 1)
                    radius = min(2.0, 0.5 * gap_len)
                    helix_offset = radius * (np.cos(angle) * perp + np.sin(angle) * perp2)
                    dampen = np.sin(np.pi * w)
                    base += helix_offset * dampen
            result[i] = base
        elif prev_v >= 0:
            if prev_v > 0 and not np.isnan(result[prev_v - 1, 0]):
                d = result[prev_v] - result[prev_v - 1]
                d = d / (np.linalg.norm(d) + 1e-12) * 5.95
            else:
                d = np.array([5.95, 0, 0])
            result[i] = result[prev_v] + d * (i - prev_v)
        elif next_v >= 0:
            if next_v < qlen - 1 and not np.isnan(result[next_v + 1, 0]):
                d = result[next_v + 1] - result[next_v]
                d = d / (np.linalg.norm(d) + 1e-12) * 5.95
            else:
                d = np.array([5.95, 0, 0])
            result[i] = result[next_v] - d * (next_v - i)
        else:
            result[i] = [i * 5.95, 0, 0]
    
    return np.nan_to_num(result), coverage


def _helical_chain(length, spacing=5.95, twist_deg=32.7, radius=2.0):
    coords = np.zeros((length, 3), dtype=np.float64)
    twist_rad = np.deg2rad(twist_deg)
    rise = spacing * 0.47
    for i in range(length):
        angle = i * twist_rad
        coords[i, 0] = radius * np.cos(angle)
        coords[i, 1] = radius * np.sin(angle)
        coords[i, 2] = i * rise
    return coords

print("Coordinate transfer loaded.")

In [ ]:
# ============================================================
# Coverage-Weighted Template Retrieval
# ============================================================
def find_templates_for_chain(chain_seq, top_n=ALIGN_TOP, length_tolerance=0.50):
    """Find best templates for a single chain sequence.
    Uses coverage-weighted scoring that accounts for both sequence similarity
    and how well the template covers the query."""
    qlen = len(chain_seq)
    qkm = kmer_set_2bit(chain_seq, KMER_K)
    qkm_len = len(qkm)
    
    if len(TRAIN_IDS) == 0:
        return []
    
    maxL = np.maximum(TRAIN_LENS, qlen)
    keep = (np.abs(TRAIN_LENS - qlen) / maxL) <= length_tolerance
    idxs = np.where(keep)[0]
    if idxs.size < 10:
        keep = (np.abs(TRAIN_LENS - qlen) / maxL) <= 0.70
        idxs = np.where(keep)[0]
    if idxs.size == 0:
        diffs = np.abs(TRAIN_LENS - qlen)
        idxs = np.argsort(diffs)[:PREFILTER_TOP]
    
    scored = []
    for i in idxs:
        tkm = TRAIN_KMERS[i]
        if qkm_len == 0 or len(tkm) == 0:
            jac = 0.0
        else:
            inter = len(qkm & tkm)
            union = qkm_len + len(tkm) - inter
            jac = inter / union if union else 0.0
        len_ratio = min(qlen, TRAIN_LENS[i]) / max(qlen, TRAIN_LENS[i])
        valid_frac = TRAIN_VALID_FRAC[i]
        composite = jac * np.sqrt(len_ratio) * valid_frac
        scored.append((composite, jac, int(i)))
    
    scored.sort(key=lambda x: x[0], reverse=True)
    top_candidates = scored[:PREFILTER_TOP]
    
    NW_CELL_LIMIT = 5_000_000
    sims = []
    for composite, jac, i in top_candidates:
        t_seq = TRAIN_SEQS[i]
        if qlen * len(t_seq) < NW_CELL_LIMIT:
            raw = nw_score_only(chain_seq, t_seq)
            denom = 2.0 * min(qlen, len(t_seq))
            norm = float(raw / denom) if denom > 0 else jac
        else:
            norm = jac
        len_ratio = min(qlen, len(t_seq)) / max(qlen, len(t_seq))
        final_score = norm * np.sqrt(len_ratio) * TRAIN_VALID_FRAC[i]
        sims.append((TRAIN_IDS[i], t_seq, final_score, norm, TRAIN_COORDS[i], i))
    
    sims.sort(key=lambda x: x[2], reverse=True)
    return sims[:top_n]

print("Template retrieval loaded.")

In [ ]:
# ============================================================
# Fragment-Based Assembly
# ============================================================
FRAGMENT_SIZE = 100
FRAGMENT_OVERLAP = 30
FRAGMENT_MIN_SIM = 0.15

def fragment_assembly(chain_seq, fallback_coords=None):
    """Assemble coordinates from overlapping fragments.
    Split chain into fragments, find best template for each,
    transfer coordinates, then stitch via Kabsch on overlaps."""
    L = len(chain_seq)
    if L <= FRAGMENT_SIZE:
        cands = find_templates_for_chain(chain_seq, top_n=5)
        if cands:
            t_id, t_seq, score, sim, t_coords, idx = cands[0]
            amap, _ = needleman_wunsch(chain_seq, t_seq)
            coords, cov = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
            return coords, cov
        return _helical_chain(L), 0.0
    
    step = FRAGMENT_SIZE - FRAGMENT_OVERLAP
    fragments = []
    pos = 0
    while pos < L:
        end = min(pos + FRAGMENT_SIZE, L)
        if L - end < FRAGMENT_OVERLAP and end < L:
            end = L
        fragments.append((pos, end))
        if end >= L:
            break
        pos += step
    
    frag_coords = []
    frag_coverages = []
    for frag_start, frag_end in fragments:
        frag_seq = chain_seq[frag_start:frag_end]
        cands = find_templates_for_chain(frag_seq, top_n=5, length_tolerance=0.60)
        if cands and cands[0][2] > FRAGMENT_MIN_SIM:
            t_id, t_seq, score, sim, t_coords, idx = cands[0]
            amap, _ = needleman_wunsch(frag_seq, t_seq)
            coords, cov = transfer_coordinates(frag_seq, t_seq, t_coords, amap)
            frag_coords.append(coords)
            frag_coverages.append(cov)
        else:
            frag_coords.append(None)
            frag_coverages.append(0.0)
    
    result = np.full((L, 3), np.nan, dtype=np.float64)
    
    placed_mask = np.zeros(L, dtype=bool)
    
    for fi, (frag_start, frag_end) in enumerate(fragments):
        if frag_coords[fi] is None:
            continue
        fc = frag_coords[fi]
        flen = frag_end - frag_start
        
        if fi == 0:
            result[frag_start:frag_end] = fc
            placed_mask[frag_start:frag_end] = True
        else:
            overlap_start = frag_start
            prev_end = fragments[fi-1][1]
            overlap_end = min(prev_end, frag_end)
            overlap_len = overlap_end - overlap_start
            
            if overlap_len >= 4 and placed_mask[overlap_start:overlap_end].sum() >= 4:
                ref_pts = result[overlap_start:overlap_end]
                new_pts = fc[:overlap_len]
                valid = ~(np.isnan(ref_pts[:, 0]) | np.isnan(new_pts[:, 0]))
                if valid.sum() >= 4:
                    try:
                        R, t = kabsch_superpose(new_pts[valid], ref_pts[valid])
                        fc_aligned = (fc @ R.T) + t
                        result[overlap_end:frag_end] = fc_aligned[overlap_len:]
                        placed_mask[overlap_end:frag_end] = True
                        for k in range(overlap_start, overlap_end):
                            if placed_mask[k] and not np.isnan(fc_aligned[k - frag_start, 0]):
                                w = (k - overlap_start) / max(overlap_len - 1, 1)
                                result[k] = (1 - w) * result[k] + w * fc_aligned[k - frag_start]
                        continue
                    except Exception:
                        pass
            
            for k in range(frag_start, frag_end):
                if not placed_mask[k]:
                    result[k] = fc[k - frag_start]
                    placed_mask[k] = True
    
    if fallback_coords is not None:
        for i in range(L):
            if np.isnan(result[i, 0]):
                result[i] = fallback_coords[i]
    else:
        for i in range(L):
            if np.isnan(result[i, 0]):
                result[i] = [i * 5.95, 0, 0]
    
    coverage = placed_mask.sum() / L
    return np.nan_to_num(result), coverage

print("Fragment assembly loaded.")

In [ ]:
# ============================================================
# Stoichiometry and Chain Handling
# ============================================================
def parse_fasta(fasta_content):
    out = {}
    cur = None
    seq_parts = []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if cur is not None:
                out[cur] = "".join(seq_parts)
            cur = line[1:].split()[0]
            seq_parts = []
        else:
            seq_parts.append(line.replace(" ", ""))
    if cur is not None:
        out[cur] = "".join(seq_parts)
    return out


def parse_stoichiometry(stoich):
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    out = []
    for part in str(stoich).split(";"):
        parts = part.split(":")
        if len(parts) == 2:
            out.append((parts[0].strip(), int(parts[1])))
    return out


def get_chain_info(row):
    """Return list of (start, end, chain_seq) for each chain in the complex."""
    seq = row["sequence"]
    stoich = row.get("stoichiometry", "")
    all_seq = row.get("all_sequences", "")
    
    if pd.isna(stoich) or pd.isna(all_seq) or str(stoich).strip() == "" or str(all_seq).strip() == "":
        return [(0, len(seq), seq)]
    
    try:
        chain_dict = parse_fasta(all_seq)
        order = parse_stoichiometry(stoich)
        chains = []
        pos = 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq), seq)]
            for _ in range(cnt):
                L = len(base)
                chains.append((pos, pos + L, base))
                pos += L
        if pos != len(seq):
            return [(0, len(seq), seq)]
        return chains
    except Exception:
        return [(0, len(seq), seq)]


test_chain_info = {}
for _, r in test_seqs.iterrows():
    test_chain_info[r["target_id"]] = get_chain_info(r)

print(f"Chain info computed for {len(test_chain_info)} test targets")
for tid, chains in list(test_chain_info.items())[:5]:
    total_res = sum(e-s for s,e,_ in chains)
    unique_seqs = len(set(cs for _,_,cs in chains))
    print(f"  {tid}: {len(chains)} chain(s), {unique_seqs} unique, total {total_res} residues")

In [ ]:
# ============================================================
# RNA Structural Constraints (enhanced)
# ============================================================
def adaptive_rna_constraints(coordinates, segments, confidence=1.0, passes=3):
    coords = coordinates.copy()
    strength = 0.75 * (1.0 - min(confidence, 0.90))
    strength = max(strength, 0.03)
    
    for _ in range(passes):
        for (s, e) in segments:
            X = coords[s:e]
            L = e - s
            if L < 3:
                continue
            
            d = X[1:] - X[:-1]
            dist = np.linalg.norm(d, axis=1) + 1e-6
            target = 5.95
            scale = (target - dist) / dist
            adj = (d * scale[:, None]) * (0.25 * strength)
            X[:-1] -= adj
            X[1:] += adj
            
            if L >= 5:
                d2 = X[2:] - X[:-2]
                dist2 = np.linalg.norm(d2, axis=1) + 1e-6
                target2 = 10.2
                scale2 = (target2 - dist2) / dist2
                adj2 = (d2 * scale2[:, None]) * (0.12 * strength)
                X[:-2] -= adj2
                X[2:] += adj2
            
            if L >= 5:
                lap = 0.5 * (X[:-2] + X[2:]) - X[1:-1]
                X[1:-1] += (0.08 * strength) * lap
            
            if L >= 25:
                k = min(L, 200)
                idx = np.linspace(0, L - 1, k).astype(int) if k < L else np.arange(L)
                P = X[idx]
                diff = P[:, None, :] - P[None, :, :]
                distm = np.linalg.norm(diff, axis=2) + 1e-6
                sep = np.abs(idx[:, None] - idx[None, :])
                mask = (sep > 2) & (distm < 3.2)
                if np.any(mask):
                    force = (3.2 - distm) / distm
                    vec = (diff * force[:, :, None] * mask[:, :, None]).sum(axis=1)
                    X[idx] += (0.02 * strength) * vec
            
            coords[s:e] = X
    
    return coords

print("Structural constraints loaded.")

In [ ]:
# ============================================================
# Template Blending with Kabsch
# ============================================================
def blend_templates(query_seq, templates, weights):
    """Blend multiple template coordinate transfers using Kabsch superposition."""
    qlen = len(query_seq)
    if not templates:
        return _helical_chain(qlen)
    
    transfers = []
    for t_seq, t_coords, amap, sim in templates:
        coords, cov = transfer_coordinates(query_seq, t_seq, t_coords, amap)
        transfers.append(coords)
    
    if len(transfers) == 1:
        return transfers[0]
    
    ref = transfers[0]
    aligned = [ref]
    
    for k in range(1, len(transfers)):
        other = transfers[k]
        valid = ~(np.isnan(ref[:, 0]) | np.isnan(other[:, 0]))
        n_valid = valid.sum()
        if n_valid >= 4:
            try:
                R, t = kabsch_superpose(other[valid], ref[valid])
                other_aligned = (other @ R.T) + t
                aligned.append(other_aligned)
            except Exception:
                aligned.append(other)
        else:
            aligned.append(other)
    
    w = np.array(weights[:len(aligned)], dtype=np.float64)
    w = w / (w.sum() + 1e-12)
    result = np.zeros((qlen, 3), dtype=np.float64)
    for k in range(len(aligned)):
        result += w[k] * aligned[k]
    
    return result

print("Template blending loaded.")

In [ ]:
# ============================================================
# TM-score Computation
# ============================================================
def compute_tm_score(pred_coords, true_coords):
    valid = ~(np.isnan(true_coords[:, 0]) | np.isnan(pred_coords[:, 0]))
    if valid.sum() < 3:
        return 0.0
    
    pred = pred_coords[valid]
    true = true_coords[valid]
    L = len(true)
    
    if L < 15:
        d0 = 0.5
    else:
        d0 = 0.6 * np.sqrt(L - 0.5) - 2.5
        d0 = max(d0, 0.5)
    
    try:
        R, t = kabsch_superpose(pred, true)
        pred_aligned = (pred @ R.T) + t
    except Exception:
        pred_aligned = pred
    
    distances = np.linalg.norm(pred_aligned - true, axis=1)
    tm = np.sum(1.0 / (1.0 + (distances / d0) ** 2)) / L
    
    return float(tm)


def compute_best_of_5_tm(all_preds, true_coords):
    best = 0.0
    for pred in all_preds:
        tm = compute_tm_score(pred, true_coords)
        best = max(best, tm)
    return best

print("TM-score computation loaded.")

In [ ]:
# ============================================================
# Per-Chain Multi-Strategy Prediction Pipeline
# ============================================================
def predict_single_chain(chain_seq, chain_id_str, n_predictions=5):
    """Predict structures for a single chain using multiple strategies."""
    L = len(chain_seq)
    cands = find_templates_for_chain(chain_seq, top_n=ALIGN_TOP)
    predictions = []
    
    if not cands:
        for _ in range(n_predictions):
            predictions.append(_helical_chain(L))
        return predictions
    
    aligned_templates = []
    for c_id, c_seq, c_score, c_sim, c_coords, c_idx in cands[:min(10, len(cands))]:
        amap, score = needleman_wunsch(chain_seq, c_seq)
        aligned_templates.append((c_id, c_seq, c_coords, amap, c_sim, score))
    
    best_sim = aligned_templates[0][4] if aligned_templates else 0.0
    use_fragments = (L > 200 and best_sim < 0.40)
    
    for i in range(n_predictions):
        if i == 0:
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[0]
            X, cov = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
        
        elif i == 1 and use_fragments:
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[0]
            fallback, _ = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
            X, cov = fragment_assembly(chain_seq, fallback_coords=fallback)
            sim = best_sim
        
        elif i == 1 and len(aligned_templates) >= 2:
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[1]
            X, cov = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
        
        elif i == 2 and len(aligned_templates) >= 3:
            blend_input = []
            blend_weights = []
            for j in range(min(3, len(aligned_templates))):
                _, t_seq, t_coords, amap, sim, _ = aligned_templates[j]
                blend_input.append((t_seq, t_coords, amap, sim))
                blend_weights.append(max(sim, 0.01))
            X = blend_templates(chain_seq, blend_input, blend_weights)
            sim = aligned_templates[0][4]
        
        elif i == 3:
            idx = min(3, len(aligned_templates) - 1)
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[idx]
            X, cov = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
        
        elif i == 4 and use_fragments:
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[min(1, len(aligned_templates)-1)]
            fallback, _ = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
            X, cov = fragment_assembly(chain_seq, fallback_coords=fallback)
            seed = (zlib.adler32(f"{chain_id_str}_frag4".encode()) & 0xFFFFFFFF)
            rng = np.random.default_rng(seed)
            X += rng.normal(0, 0.3, X.shape)
            sim = best_sim
        
        elif i == 4 and len(aligned_templates) >= 5:
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[4]
            X, cov = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
        
        else:
            idx = min(i, len(aligned_templates) - 1)
            _, t_seq, t_coords, amap, sim, _ = aligned_templates[idx]
            X, cov = transfer_coordinates(chain_seq, t_seq, t_coords, amap)
            seed = (zlib.adler32(f"{chain_id_str}_{i}".encode()) & 0xFFFFFFFF)
            rng = np.random.default_rng(seed)
            noise_std = max(0.3, (0.5 - sim) * 2.0)
            X += rng.normal(0, noise_std, X.shape)
        
        predictions.append(X)
    
    return predictions


def predict_complex(tid, full_seq, chain_info, n_predictions=5):
    """Predict the full complex by predicting each chain independently."""
    L = len(full_seq)
    all_predictions = [np.zeros((L, 3), dtype=np.float64) for _ in range(n_predictions)]
    
    chain_cache = {}
    
    for ci, (start, end, chain_seq) in enumerate(chain_info):
        if chain_seq in chain_cache:
            chain_preds = chain_cache[chain_seq]
        else:
            chain_id = f"{tid}_chain{ci}"
            chain_preds = predict_single_chain(chain_seq, chain_id, n_predictions=n_predictions)
            chain_cache[chain_seq] = chain_preds
        
        for pi in range(n_predictions):
            pred_coords = chain_preds[pi]
            chain_len = end - start
            
            if ci > 0 and chain_seq in chain_cache:
                seed = (zlib.adler32(f"{tid}_{ci}_{pi}".encode()) & 0xFFFFFFFF)
                rng = np.random.default_rng(seed)
                axis = rng.normal(size=3)
                axis = axis / (np.linalg.norm(axis) + 1e-12)
                angle = rng.uniform(0, 2 * np.pi)
                c, s = np.cos(angle), np.sin(angle)
                x, y, z = axis
                C = 1.0 - c
                R = np.array([
                    [c + x*x*C, x*y*C - z*s, x*z*C + y*s],
                    [y*x*C + z*s, c + y*y*C, y*z*C - x*s],
                    [z*x*C - y*s, z*y*C + x*s, c + z*z*C]
                ])
                centroid = pred_coords.mean(axis=0)
                rotated = (pred_coords - centroid) @ R.T + centroid
                offset = rng.normal(0, 15, size=3)
                pred_coords = rotated + offset
            
            all_predictions[pi][start:end] = pred_coords[:chain_len]
    
    segments = [(s, e) for s, e, _ in chain_info]
    for pi in range(n_predictions):
        best_sim = 0.5
        all_predictions[pi] = adaptive_rna_constraints(
            all_predictions[pi], segments, confidence=best_sim, passes=3
        )
    
    return all_predictions

print("Per-chain prediction pipeline loaded.")

In [ ]:
# ============================================================
# Validation (if available)
# ============================================================
if HAS_VAL:
    print("\n" + "=" * 60)
    print("Running validation...")
    print("=" * 60)
    
    val_chain_info = {}
    for _, r in val_seqs.iterrows():
        val_chain_info[r["target_id"]] = get_chain_info(r)
    
    val_coords_dict = process_labels(val_labels)
    
    val_tm_scores = []
    t0 = time.time()
    for _, r in val_seqs.iterrows():
        tid = r["target_id"]
        seq = r["sequence"]
        chains = val_chain_info.get(tid, [(0, len(seq), seq)])
        
        if tid not in val_coords_dict:
            continue
        true_coords = val_coords_dict[tid]
        if len(true_coords) != len(seq):
            continue
        
        preds = predict_complex(tid, seq, chains, n_predictions=5)
        tm = compute_best_of_5_tm(preds, true_coords)
        val_tm_scores.append((tid, tm, len(seq)))
    
    val_time = time.time() - t0
    
    if val_tm_scores:
        scores = [s[1] for s in val_tm_scores]
        print(f"\nValidation results ({len(val_tm_scores)} targets, {val_time:.1f}s):")
        print(f"  Mean TM-score: {np.mean(scores):.4f}")
        print(f"  Median TM-score: {np.median(scores):.4f}")
        print(f"  Min: {np.min(scores):.4f}, Max: {np.max(scores):.4f}")
        print(f"  Scores > 0.3: {sum(1 for s in scores if s > 0.3)}/{len(scores)}")
        print(f"  Scores > 0.4: {sum(1 for s in scores if s > 0.4)}/{len(scores)}")
        print(f"\n  Per-target (top 10):")
        for tid, tm, L in sorted(val_tm_scores, key=lambda x: -x[1])[:10]:
            print(f"    {tid}: TM={tm:.4f} (L={L})")
        print(f"\n  Per-target (bottom 5):")
        for tid, tm, L in sorted(val_tm_scores, key=lambda x: x[1])[:5]:
            print(f"    {tid}: TM={tm:.4f} (L={L})")
    else:
        print("No valid validation targets found")
else:
    print("Skipping validation (no validation set available)")

In [ ]:
# ============================================================
# Generate Predictions for All Test Targets
# ============================================================
print(f"\nGenerating predictions for {len(test_seqs)} test targets...")
start_time = time.time()

dfs = []

for idx, r in enumerate(test_seqs.itertuples(index=False)):
    tid = r.target_id
    seq = r.sequence
    L = len(seq)
    chains = test_chain_info.get(tid, [(0, L, seq)])
    
    t0 = time.time()
    preds = predict_complex(tid, seq, chains, n_predictions=5)
    elapsed = time.time() - t0
    
    n_chains = len(chains)
    if idx < 5 or idx % 10 == 0:
        print(f"  [{idx+1}/{len(test_seqs)}] {tid} (L={L}, chains={n_chains}) -> {elapsed:.1f}s")
    
    data = {
        "ID": [f"{tid}_{j}" for j in range(1, L + 1)],
        "resname": list(seq),
        "resid": np.arange(1, L + 1, dtype=np.int32),
    }
    for i in range(5):
        data[f"x_{i+1}"] = preds[i][:, 0].astype(np.float32)
        data[f"y_{i+1}"] = preds[i][:, 1].astype(np.float32)
        data[f"z_{i+1}"] = preds[i][:, 2].astype(np.float32)
    
    dfs.append(pd.DataFrame(data))

total_time = time.time() - start_time
print(f"\nAll predictions generated in {total_time:.1f}s")

In [ ]:
# ============================================================
# Build and Save Submission
# ============================================================
sub = pd.concat(dfs, ignore_index=True)

cols = ["ID", "resname", "resid"] + [f"{c}_{i}" for i in range(1, 6) for c in ["x", "y", "z"]]
coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]

sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
sub = sub.fillna(0.0)

output_path = os.path.join(WORK, "submission.csv")
sub[cols].to_csv(output_path, index=False)

print(f"Submission saved to {output_path}")
print(f"Shape: {sub.shape}")
print(f"Expected: {len(sample_sub)} rows")
print(f"NaN values: {sub[coord_cols].isna().sum().sum()}")
print(f"Zero values: {(sub[coord_cols] == 0).sum().sum()}")
print()
print(sub.head())
print()
print("=" * 60)
print("SUB006 Pipeline Complete")
print("=" * 60)
print(f"  Templates used  : {len(TRAIN_IDS)}")
print(f"  Test targets    : {len(test_seqs)}")
print(f"  Submission rows : {len(sub)}")
print(f"  Total time      : {total_time:.1f}s")
print("=" * 60)